# Types

> Common types helpers for MCP tool functions

In [ ]:
#| default_exp utils.types

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Callable, TypeVar, Literal
from dataclasses import dataclass, field
from pathlib import Path
from functools import wraps
import io

from rich.table import Table
from rich.panel import Panel
from rich.console import Console
from rich.text import Text
from rich.markdown import Markdown

## Result Types

Standardized result types for tool responses.

In [ ]:
#| export
@dataclass
class ToolResult:
    """Standardized result from a tool function.
    
    All tool functions should return a dict via `.to_dict()`.
    """
    ok: bool = True
    error: str = ''
    data: Dict[str, Any] = field(default_factory=dict)
    pretty: str = ''
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dict for MCP response."""
        result = {'ok': self.ok}
        if self.error:
            result['error'] = self.error
        result.update(self.data)
        if self.pretty:
            result['pretty'] = self.pretty
        return result
    
    @classmethod
    def success(cls, pretty: str = '', **data) -> 'ToolResult':
        """Create a success result."""
        return cls(ok=True, data=data, pretty=pretty)
    
    @classmethod
    def failure(cls, error: str) -> 'ToolResult':
        """Create a failure result."""
        return cls(ok=False, error=error)

## Lint Issue Types

Standardized types for lint/validation issues.

In [ ]:
#| export
@dataclass
class Issue:
    """A lint or validation issue."""
    rule: str
    message: str
    file: str = ''
    notebook: str = ''
    cell: int = -1
    line: int = -1
    suggestion: str = ''
    
    def to_dict(self) -> Dict[str, Any]:
        """Convert to dict."""
        d = {'rule': self.rule, 'message': self.message}
        if self.file:
            d['file'] = self.file
        if self.notebook:
            d['notebook'] = self.notebook
        if self.cell >= 0:
            d['cell'] = self.cell
        if self.line >= 0:
            d['line'] = self.line
        if self.suggestion:
            d['suggestion'] = self.suggestion
        return d

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install Types

Types for MCP installer operations.

In [ ]:
#| export
InstallAction = Literal["created", "updated", "skipped", "dry_run", "error"]

In [ ]:
#| export
@dataclass
class InstallResult:
    """Result of an installation operation.
    
    Attributes
    ----------
    success : bool
        Whether the operation succeeded.
    message : str
        Human-readable result message.
    config_path : Path
        Path to the configuration file.
    action : InstallAction
        What action was taken.
    provider : str
        Provider name (e.g., 'claude', 'vscode').
    details : dict
        Additional details about the operation.
    """
    success: bool
    message: str
    config_path: Path
    action: InstallAction
    provider: str = ''
    details: dict = field(default_factory=dict)
    
    def to_dict(self) -> dict:
        """Convert to dictionary representation."""
        return {
            'success': self.success,
            'message': self.message,
            'config_path': str(self.config_path),
            'action': self.action,
            'provider': self.provider,
            'details': self.details
        }

In [ ]:
#| export
def get_python_path() -> str:
    """Get the current Python interpreter path."""
    import sys
    return sys.executable

In [ ]:
#| export
@dataclass
class MCPServerConfig:
    """Configuration for an MCP server.
    
    Attributes
    ----------
    name : str
        Server name (e.g., 'nbdev').
    command : str
        Command to run the server.
    args : list[str]
        Command arguments.
    transport : str
        Transport type ('stdio' or 'http').
    env : dict
        Environment variables.
    """
    name: str
    command: str
    args: list[str] = field(default_factory=list)
    transport: str = 'stdio'
    env: dict = field(default_factory=dict)
    
    @classmethod
    def for_nbdev(cls, python_path: Optional[str] = None) -> 'MCPServerConfig':
        """Create config for nbdev-mcp server."""
        python = python_path or get_python_path()
        return cls(
            name='nbdev',
            command=python,
            args=['-m', 'nbdev_mcp', '--transport', 'stdio'],
            transport='stdio'
        )
    
    def to_vscode_format(self) -> dict:
        """Convert to VS Code MCP format."""
        return {
            'type': self.transport,
            'command': self.command,
            'args': self.args
        }
    
    def to_claude_format(self) -> dict:
        """Convert to Claude Code format."""
        return {
            'type': self.transport,
            'command': self.command,
            'args': self.args,
            'env': self.env
        }
    
    def to_toml_section(self) -> str:
        """Convert to TOML section string."""
        args_str = ', '.join(f'"{a}"' for a in self.args)
        return f'''[mcp_servers.{self.name}]
command = "{self.command}"
args = [{args_str}]
'''